# 07 — KNN e matrizes esparsas

**DATA SCIENCE KNOWLEDGE BASE**  
**Área:** `09_recommender_systems` — Sistemas de Recomendação  
**Capítulo:** 07 de 16  

> Material didático autoral revisado a partir dos notebooks de estudo. Execute as células em sequência, de cima para baixo, em um kernel reiniciado. Consulte as fontes históricas somente para estudo, preservando sua proveniência.

## Objetivos de aprendizagem
- Comprimir matriz de interações em CSR
- Buscar vizinhos sem densificar toda a matriz

## Materiais complementares

- [Guia Técnico](reference_materials/07_knn_matrizes_esparsas.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_07_knn_matrizes_esparsas.pdf)

**Navegação:** os links são relativos à pasta deste notebook. A versão editorial desses PDFs será revalidada após a consolidação científica dos notebooks.

## Fundamentação conceitual
Uma matriz CSR armazena apenas entradas observadas. `NearestNeighbors` aceita dados esparsos e permite consultar vizinhos sem calcular todas as similaridades par-a-par.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Codificação esparsa

In [2]:
from sklearn.neighbors import NearestNeighbors
u_codes,u_ids=pd.factorize(ratings.userId)
i_codes,i_ids=pd.factorize(ratings.movieId)
R=sparse.csr_matrix((ratings.rating,(u_codes,i_codes)),shape=(len(u_ids),len(i_ids)))
print('Shape:',R.shape,'nnz:',R.nnz,'densidade:',R.nnz/(R.shape[0]*R.shape[1]))

Shape: (610, 9724) nnz: 100836 densidade: 0.016999683055613623


**Interpretação.** CSR é apropriada para armazenamento de matrizes muito vazias.

## KNN dos usuários

In [3]:
knn=NearestNeighbors(metric='cosine',algorithm='brute',n_neighbors=6).fit(R)
dist,inds=knn.kneighbors(R[0],n_neighbors=6)
print([(int(u_ids[j]),round(float(d),3)) for j,d in zip(inds[0],dist[0])])

[(1, 0.0), (266, 0.643), (313, 0.648), (368, 0.655), (57, 0.655), (91, 0.665)]


**Interpretação.** A distância de cosseno é 1 menos a similaridade de cosseno.

## Comparar memória

In [4]:
print('Bytes arrays CSR:',R.data.nbytes+R.indices.nbytes+R.indptr.nbytes)
print('Bytes matriz densa float64:',R.shape[0]*R.shape[1]*8)

Bytes arrays CSR: 1212476
Bytes matriz densa float64: 47453120


**Interpretação.** Matrizes densas podem consumir memória desnecessariamente.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** CSR e memória. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **CSR e memória** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **CSR e memória** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — CSR: memória e identificação correta de vizinhos

CSR armazena índices, ponteiros de linha e valores não nulos. A eficiência depende de densidade e tipos numéricos; comparar apenas os bytes do array `data` subestima seu custo. Na consulta KNN, remova o próprio usuário e diferencie **distância cosseno** de similaridade (`distância = 1 − similaridade` em vetores não nulos).

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [5]:
# Checagem de memória CSR vs. representação densa equivalente
mem_csr = R.data.nbytes + R.indices.nbytes + R.indptr.nbytes
mem_densa = R.shape[0] * R.shape[1] * R.dtype.itemsize
print(f'CSR: {mem_csr/1024**2:.2f} MiB; matriz densa equivalente: {mem_densa/1024**2:.2f} MiB')
assert R.nnz <= R.shape[0]*R.shape[1]

CSR: 1.16 MiB; matriz densa equivalente: 45.25 MiB


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
Zeros continuam representando ausências no algoritmo, não rejeições explícitas.

## Exercícios
Mostre por que `n_neighbors=6` retorna o próprio usuário nas consultas.

## Soluções comentadas
A primeira distância usualmente é zero; descarte o próprio usuário ao recomendar.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [6]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"usuario_vizinho": [int(u_ids[j]) for j in inds[0]],
              "distancia_cosseno": [float(x) for x in dist[0]]}).assign(
    usuario_consulta=int(u_ids[0])).to_csv(out / "07_knn_usuarios.csv", index=False)
print('Outputs do capítulo 07 salvos em:', out.resolve())


Outputs do capítulo 07 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
